In [ ]:
from Geometry import * 
from PINNAX import *

import matplotlib.pyplot as plt

I1 = Interval(0.,0.5)
I2 = Interval(0.5,1.)

J = GeometryXTime(I1.union(I2), 0,1)
J.generate_basin(10000, 10000)
S,B,IC = J.get_sample(1000,200,200).values()

plt.scatter(S[:,0], S[:,1])
plt.scatter(B[:,0], B[:,1])
plt.scatter(IC[:,0], IC[:,1])
plt.show()

: 

In [ ]:

d1 = Disk([0,0], 1)
d2 = Disk([0,0], 0.5)
disk = d1.difference(d2)
disk.generate_basin(1000, 1000)
in_pts = disk.get_sample(1000, 10)['domain']
plt.scatter(in_pts[:,0], in_pts[:,1])
bnd_pts = disk.random_boundary_points(1000)
plt.scatter(bnd_pts[:,0], bnd_pts[:,1])
plt.show()


In [ ]:

def bc1(x):
    return jnp.zeros_like(x)

def bc2(x):
    return jnp.ones_like(x)

def where(x, dom):
    return dom.on_boundary(x)

BCs = [Condition(fn=bc1, dims=(0), bool_cond_fn=lambda x: where(x, d1)), Condition(fn=bc2, dims=(0), bool_cond_fn=lambda x: where(x, d2))]


In [ ]:

import jax
import jax.numpy as jnp
from jax import random, grad, jit, vmap
import optax

def init_mlp_params(layer_sizes, key):
    keys = random.split(key, len(layer_sizes))
    params = []
    for i in range(len(layer_sizes) - 1):
        w_key, b_key = random.split(keys[i])
        w = random.normal(w_key, (layer_sizes[i], layer_sizes[i+1])) * jnp.sqrt(2.0 / layer_sizes[i])
        b = random.normal(b_key, (layer_sizes[i+1],)) * 0.1
        params.append((w, b))
    return params

def apply_fn(params, x):
    activations = x

    for w, b in params[:-1]:
        outputs = jnp.dot(activations, w) + b
        activations = jnp.tanh(outputs)

    final_w, final_b = params[-1]
    out = jnp.dot(activations, final_w) + final_b
    return out

batched_apply_fn = vmap(apply_fn, in_axes=(None, 0), out_axes=0)

params_init = init_mlp_params([2, 50, 50, 1], random.PRNGKey(42))
prblm=CauchyProblem(disk, jax.jit(laplace_2d, static_argnames=['apply_fn']), BCs)
disk.generate_basin(10000, 10000)
prblm.buffer_bc()


In [ ]:

def pinn_loss(params, apply_fn):
    pde_loss = prblm.res_mse(apply_fn, params, 1000)
    bc_loss = prblm.bc_mse(apply_fn, params, [200, 200], from_buff=True)

    return pde_loss + bc_loss


In [ ]:

from tqdm import tqdm

grad_loss = grad(pinn_loss, argnums=0)
optimizer = optax.adam(learning_rate=1e-2)
opt_state = optimizer.init(params_init)
params_dnn = params_init

for epoch in tqdm(range(2000)):
    grads = grad_loss(params_dnn, apply_fn)
    updates, opt_state = optimizer.update(grads, opt_state)
    params_dnn = optax.apply_updates(params_dnn, updates)


In [ ]:
pnts = disk.get_sample(10000,10,10)['domain']
res = apply_fn(params_dnn, jnp.array(pnts))

plt.scatter(pnts[:,0], pnts[:,1], c=res[:,0], cmap='turbo')
plt.show()